# 05 Lemmatize

Reads `processed/04_abbreviations.csv` and reduces each word to its dictionary form with spaCy ("legs" becomes "leg", "walking" becomes "walk", "ran" becomes "run"). Each answer is lemmatized as a whole, so the tagger can use the neighbouring words to decide whether "saw" is a noun or a verb.

Punctuation is dropped here (the later steps work one word at a time). Numbers and words are kept.

Output: `processed/05_lemmatized.csv` with
- `answer_lemmatized`: the lemmas, in order, separated by spaces.
- `tokens`: `word|lemma|pos` for every word, separated by spaces, so the original word is still there for the final file.

In [1]:
import pandas as pd
import spacy
from collections import Counter

# the parser and entity recogniser are not needed
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
print(nlp.pipe_names)

['tok2vec', 'tagger', 'attribute_ruler', 'lemmatizer']


In [2]:
data = pd.read_csv("processed/04_abbreviations.csv", dtype=str, keep_default_na=False)
print(f"{len(data):,} rows")

185,494 rows


In [3]:
# nouns that are only used in the plural: the singular means something else, so keep them as written
KEEP_AS_WRITTEN = {
    "clothes", "pants", "glasses", "jeans", "shorts", "scissors", "stairs", "species", "politics",
    "mathematics", "athletics", "ethics", "physics", "economics", "goggles", "tights", "trousers",
    "pliers", "headquarters", "crossroads", "outskirts", "belongings", "savings", "earnings",
}

def lemma_of(t):
    return t.text.lower() if t.text.lower() in KEEP_AS_WRITTEN else t.lemma_.lower()

lemmatized, tokens = [], []
changed = Counter()

for doc in nlp.pipe(data["answer_abbreviations"], batch_size=1000):
    words = [t for t in doc if not (t.is_punct or t.is_space)]
    lemmatized.append(" ".join(lemma_of(t) for t in words))
    tokens.append(" ".join(f"{t.text}|{lemma_of(t)}|{t.pos_}" for t in words))
    for t in words:
        if lemma_of(t) != t.text:
            changed[(t.text, lemma_of(t))] += 1

data["answer_lemmatized"] = lemmatized
data["tokens"] = tokens
print(f"{sum(changed.values()):,} words changed to a different form")

286,659 words changed to a different form


The most common changes, to check them:

In [4]:
log = pd.DataFrame([(a, b, n) for (a, b), n in changed.items()], columns=["word", "lemma", "count"])
log = log.sort_values("count", ascending=False)
log.to_csv("processed/05_lemma_changes.csv", index=False)
log.head(40)

,word,lemma,count
18,is,be,34401
30,used,use,10978
134,has,have,6807
169,are,be,5249
131,things,thing,4164
92,made,make,3306
255,being,be,2200
377,makes,make,1757
44,animals,animal,1613
172,found,find,1590


In [5]:
pd.set_option("display.max_colwidth", 100)
sample = data.sample(8, random_state=3)[["answer_abbreviations", "answer_lemmatized"]]
sample

,answer_abbreviations,answer_lemmatized
99253,is a sheep.,be a sheep
81525,not evil,not evil
87797,"abandon, lie to, deceive. not loyal",abandon lie to deceive not loyal
175889,"fear, running, horror movies",fear running horror movie
38487,to tell the truth of something that one has done,to tell the truth of something that one have do
39967,plead;borrow;desperation,plead;borrow;desperation
174260,dirt,dirt
148634,against a certain subject one sided opinionated,against a certain subject one side opinionate


In [6]:
out = data[["row_id", "subject_id", "cue", "answer_lemmatized", "tokens"]]
out.to_csv("processed/05_lemmatized.csv", index=False)
print(f"Saved {len(out):,} rows to processed/05_lemmatized.csv")

Saved 185,494 rows to processed/05_lemmatized.csv
